# Intelligent Agents: Reflex-Based Agents for GridHunt

Student Name: [Mai Thu Hường]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: Undergrads 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and the hunter and the monster move on this grid. A hunter catches the monster if she/he moves on the same square the monster currently occupies. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunter are randomly placed in the arena environment. The hunter and the monster
    can move around, but cannot leave the arena.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena but only gets the monster location when it stops and listens.

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [2]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_agent_function, 
                      monster_agent_function, 
                      n = 30, max_steps = 100, 
                      visualize = False, animation = False):
    """
    Simulate an arena where a hunter agent tries to catch a monster agent.

    Parameters:
    hunter_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_pos = np.random.randint(0, n-1, size=2)
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step+1}: Hunter is at '{hunter_pos}'; Monster is at '{monster_pos}'")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_pos[0], hunter_pos[1]] = 'H'
        if np.array_equal(hunter_pos, monster_pos):
            arena[monster_pos[0], monster_pos[1]] = '*'
        print("\n".join("".join(row) for row in arena))

    # run the environment
    hunter_action = None  # Initialize hunter_action to None for the first step
    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_pos, monster_pos):
            if visualize:
                print(f"Hunter caught the monster in {step} steps!")
            return step
        
        # Get next move from monster agent function
        monster_action = monster_agent_function(hunter_pos, monster_pos)
        
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function. The monster's position is only provided 
        # if the hunter chose 'listen' as the previous action.
        if hunter_action == 'listen':
            monster_pos_instrument = monster_pos
        else: 
            monster_pos_instrument = None
        
        hunter_action = hunter_agent_function(hunter_pos, monster_pos_instrument)
        
        if hunter_action == 'listen':
            pass
        elif hunter_action == 'teleport':
            hunter_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_pos = move(hunter_action, hunter_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter chose action '{hunter_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Hunter failed to catch the monster in {max_steps} steps.")
    
    return np.nan

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [3]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(hunter_pos, monster_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [4]:
actions = ["north", "east", "west", "south", "teleport", "listen"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [5]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [6]:
simple_randomized_hunter_agent_function([0,0], None)

np.str_('north')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [7]:
arena_environment(simple_randomized_hunter_agent_function, 
                  monster_agent_function_simple, 
                  n=5, max_steps=10, visualize=True, animation=False)

Step 1: Hunter is at '[1 3]'; Monster is at '[0 3]'
...M.
...H.
.....
.....
.....
Hunter chose action 'listen'
Monster chose action 'north'


Step 2: Hunter is at '[1 3]'; Monster is at '[0 3]'
...M.
...H.
.....
.....
.....
Hunter chose action 'teleport'
Monster chose action 'north'


Step 3: Hunter is at '[2 1]'; Monster is at '[0 3]'
...M.
.....
.H...
.....
.....
Hunter chose action 'north'
Monster chose action 'stay'


Step 4: Hunter is at '[1 1]'; Monster is at '[0 3]'
...M.
.H...
.....
.....
.....
Hunter chose action 'listen'
Monster chose action 'stay'


Step 5: Hunter is at '[1 1]'; Monster is at '[0 3]'
...M.
.H...
.....
.....
.....
Hunter chose action 'north'
Monster chose action 'stay'


Step 6: Hunter is at '[0 1]'; Monster is at '[0 3]'
.H.M.
.....
.....
.....
.....
Hunter chose action 'north'
Monster chose action 'north'


Step 7: Hunter is at '[0 1]'; Monster is at '[0 3]'
.H.M.
.....
.....
.....
.....
Hunter chose action 'east'
Monster chose action 'south'


Step 8: Hunt

nan

**Note for VS Code:** View as scrollable element to see the complete output.

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $10 \times 10$ arena.  

In [8]:
steps = [arena_environment(simple_randomized_hunter_agent_function, monster_agent_function_simple, n=10, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[nan, nan, 59, 20, nan, 68, 43, nan, nan, 47, 73, nan, nan, 72, 51, nan, nan, nan, nan, 19, nan, nan, 17, 17, nan, 23, nan, nan, nan, 92, nan, 97, 19, 52, nan, nan, nan, 52, nan, nan, nan, nan, nan, nan, 43, 3, nan, nan, 5, nan, 87, nan, 11, nan, 84, 90, nan, nan, 63, 94, nan, 55, nan, nan, 11, nan, nan, nan, nan, nan, 89, 9, nan, nan, 14, nan, 79, nan, nan, nan, 55, nan, 32, 44, 6, nan, 79, nan, 82, 43, nan, nan, nan, nan, nan, nan, 19, 20, 24, nan]


We just got a list with the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results. `nan` means that the hunter was not able to catch the monster. How many times did that happen? To answer how well the hunter did when it caught the monster, we just average the numbers that are not `nan`. 

In [9]:
print (f"Hunter failed {np.sum(np.isnan(steps))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps))} successful runs: {round(np.nanmean(steps), 2)}")

Hunter failed 58 times.
Average steps to catch the monster over 42 successful runs: 46.71


This in not a very good agent. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and make it into a model-based reflex agent. It should use listen to get the monster's location and then move towards the monster. To do that it needs to remember the monster's location. This memory makes it model-based. Here is HOWTO: How to Store State Information in the Agent.

In [12]:
# here goes your agent implementation

def make_hunter_agent_function(listen_every=3):
    """
    Creates a model-based reflex hunter agent.

    State (memory) kept in the closure:
    - last_known_monster_pos: the monster's position the last time we listened
    - moves_since_listen: how many steps we've moved since that last listen

    Strategy:
    - If we don't have any memory of the monster yet, listen.
    - If our memory is "fresh" (we haven't moved too many steps since we
      listened), walk towards the remembered position.
    - If the memory is "stale" (listen_every steps have passed, or we
      already reached the remembered position but the monster wasn't there),
      listen again to refresh it.
    """
    last_known_monster_pos = None
    moves_since_listen = 0

    def hunter_agent_function(hunter_location, monster_location):
        nonlocal last_known_monster_pos, moves_since_listen

        # We just listened: the environment gave us the monster's real position.
        # Store a COPY of it (not a reference!) so it isn't changed later when
        # the environment moves the monster's numpy array in place.
        if monster_location is not None:
            last_known_monster_pos = np.array(monster_location)
            moves_since_listen = 0

        # No memory yet, or memory is too old -> listen to refresh it
        if last_known_monster_pos is None or moves_since_listen >= listen_every:
            moves_since_listen = 0
            return "listen"

        moves_since_listen += 1

        # Walk towards the remembered position: reduce the larger gap first
        row_diff = last_known_monster_pos[0] - hunter_location[0]
        col_diff = last_known_monster_pos[1] - hunter_location[1]

        if row_diff == 0 and col_diff == 0:
            # We reached the remembered spot but the monster has moved away
            return "listen"

        if abs(row_diff) >= abs(col_diff):
            return "south" if row_diff > 0 else "north"
        else:
            return "east" if col_diff > 0 else "west"

    return hunter_agent_function


# quick sanity check: ask a fresh agent for an action with no info yet
test_hunter = make_hunter_agent_function()
print(test_hunter(np.array([0, 0]), None))   # should print 'listen'


listen


In [20]:
arena_environment(make_hunter_agent_function(), 
                  monster_agent_function_simple, 
                  n=5, max_steps=10, visualize=True, animation=True)

Step 4: Hunter is at '[1 1]'; Monster is at '[1 1]'
.....
.*...
.....
.....
.....
Hunter caught the monster in 3 steps!


3

In [22]:
steps = [arena_environment(make_hunter_agent_function(), monster_agent_function_simple, n=10, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[8, 9, 6, 0, 1, 12, 6, 11, 7, 8, 11, 3, 4, 3, 11, 14, 8, 7, 4, 8, 6, 19, 6, 2, 2, 13, 10, 9, 12, 7, 6, 7, 0, 3, 6, 12, 9, 17, 6, 10, 6, 7, 3, 4, 2, 13, 12, 11, 8, 2, 10, 4, 7, 6, 19, 13, 14, 9, 6, 3, 11, 12, 10, 10, 11, 10, 17, 2, 5, 14, 20, 2, 10, 4, 10, 17, 14, 10, 10, 8, 15, 6, 10, 0, 3, 10, 12, 5, 7, 10, 5, 14, 21, 8, 10, 14, 11, 11, 15, 12]


In [23]:
print (f"Hunter failed {np.sum(np.isnan(steps))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps))} successful runs: {round(np.nanmean(steps), 2)}")

Hunter failed 0 times.
Average steps to catch the monster over 100 successful runs: 8.68


# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [11]:
# Your experimentation code goes here

## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

> Your discussion goes here

The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

> Your discussion goes here

# More Work (Optional)

Here are some ideas:

* Implement a better monster agent function and perform experiments
* Change the environment so multiple hunters can hunt the monster.
* Give the hunter a new action that shoots an arrow in a specific direction. 
  The arrow can go a maximum of 5 squares. If the hunter hits the monster, then it wins.